In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('/home/admin/Documents/98_model/src')
sys.path.append('/data01/Documents/98_model/src')
sys.path.append('/data/98_model/src')

# Import modules

In [3]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from IPython.display import display

import seaborn as sns
import xgboost as xgb
import yaml
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import KFold

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import xgboost as xgb
from tqdm import tqdm
import re
import pickle
from sklearn.metrics import roc_curve, auc
import re

from tqdm import trange
import pickle
import os
import optuna

from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import threading
import gc

from sklearn.manifold import TSNE
import umap
import dill
from matplotlib.patches import FancyArrowPatch
from sklearn.metrics import root_mean_squared_error
from optuna.visualization import plot_optimization_history
from model_v4 import CascadeModel
from plotly.io import show

/home/admin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/admin/.venv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [4]:
#optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [5]:
from utils import PROCESS_STEPS, INPUT_PROFILE, DATA_TYPES
from utils import ColumnClassifier
from utils import squeeze_list

# Environment variables

In [6]:
SHOW_DISTRIBUTION = False
PERFORM_CROSS_VALIDATION = True
PERFORM_BACKTEST = True

In [7]:
FILEPATH_PCA_MODEL_SV = 'PCA_MODEL_SV.dill'
FILEPATH_PCA_MODEL_IQC = 'PCA_MODEL_IQC.dill'
FILEPATH_PCA_MODEL_SV_IQC = 'PCA_MODEL_SV_IQC.dill'

In [8]:
model_name = 'N32S'
n_trials = 2001
lb_margin = 0.05
ub_margin = 0.05
threshold_constraint = 3

model_version = 'v4'

# Load data

In [9]:
try : 
    data_path = '/home/admin/Documents/98_model/notebooks/260714_feature_engineering_qa/feature_store_v10_n32s.parquet'
    data = pd.read_parquet(data_path)
except : 
    try : 
        data_path = '/data01/Documents/98_model/notebooks/260714_feature_engineering_qa/feature_store_v10_n32s.parquet'
        data = pd.read_parquet(data_path)
    except : 
        data_path = '/data/98_model/notebooks/260714_feature_engineering_qa/feature_store_v10_n32s.parquet'
        data = pd.read_parquet(data_path)
data

,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2B2016718,59JFB112A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1T1A5C1,M2ECOT001,2026-01-30 06:12:56,5CF1T1A5R2,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2E2063322,59JFB142A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF2A131C1,M2ECOT002,2026-02-10 14:14:12,5AF2A131R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2G2018702,59JFB162A1,5A3F209041,M2EMIX01702,2026-02-09 19:11:46,5CF1V165C1,M2ECOT001,2026-01-31 18:56:59,5CF1V165R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G382040158,59JFC082A1,5C1F225029,M2EMIX01103,2026-02-26 00:00:57,5CF2R151C1,M2ECOT001,2026-02-27 17:44:29,5CF2R151R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G3S2044730,59JFC252A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E183C1,M2ECOT001,2026-03-14 20:58:31,5CF3E183R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,None,59JFC302A1,5A4F320066,M2EMIX01802,2026-03-20 15:59:56,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,None,59JFC302A1,5C1F324051,M2EMIX01103,2026-03-24 15:31:54,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P135C1,M2ECOT001,2026-03-25 14:02:37,5CF3P135R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P164C1,M2ECOT001,2026-03-25 19:24:35,5CF3P164R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
data_path_weekly_defect = '/home/admin/Documents/98_model/data/weekly_defect_n32s.parquet'
data_path_weekly_defect_vm2 = '/data01/Documents/98_model/data/weekly_defect_n32s.parquet'
data_path_weekly_defect_vm1 = '/data/98_model/data/weekly_defect_n32s.parquet'

try : 
    weekly_score = pd.read_parquet(data_path_weekly_defect)
except : 
    try : 
        weekly_score = pd.read_parquet(data_path_weekly_defect_vm2)
    except : 
        weekly_score = pd.read_parquet(data_path_weekly_defect_vm1)

In [11]:
weekly_score.head().T

,0,1,2,3,4
date,2026.01.25-2026.01.31,2026.02.01-2026.02.07,2026.02.08-2026.02.14,2026.02.15-2026.02.21,2026.02.22-2026.02.28
week,5,6,7,8,9
Y_NFF_A,0.997249,0.996782,0.997993,0.998562,0.996932
Y_NFF_B,0.0,0.0,0.0,0.0,0.0
Y_NFF_C,0.0,0.0,0.0,0.0,0.0
Y_NFF_D,0.000045,0.000101,0.000134,0.000075,0.000164
Y_NFF_E,0.000435,0.000564,0.00048,0.000534,0.000457
Y_NFF_F,0.000007,0.0,0.0,0.0,0.0
Y_NFF_G,0.0,0.0,0.0,0.0,0.0
Y_NFF_H,0.00036,0.00022,0.000289,0.000368,0.000287


# Get column names

In [12]:
column_classifier = ColumnClassifier()
df_cols = column_classifier.transform(data=data)

In [13]:
cols_small_y = (
    df_cols.loc[lambda x :x['data_type'] == 'Small_Y', 'cols']
    .tolist()
)
cols_small_y = squeeze_list(cols_small_y)
cols_small_y = [x for x in cols_small_y if x in data.columns]
cols_small_y[:5]

['y_MES_Electrode_Coating_Anode_치수_Mismatch (Back)',
 'y_MES_Electrode_Coating_Cathode_외관_접힘 (Back)',
 'y_LQC_Electrode_Coating_Anode_Sliding(+)_Tab Top',
 'y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back',
 'y_MES_Electrode_Coating_Anode_절연 (Top)']

In [14]:
cols_sv = (
    df_cols.loc[lambda x :x['data_type'] == 'SV', 'cols']
    .tolist()
)
cols_sv = squeeze_list(cols_sv)
cols_sv[:5]

['X_SV_Electrode_Mixing_Anode_Parameter Value_2-1 Step Mixing Time',
 'X_SV_Electrode_Mixing_Cathode_Parameter Value_2-1 Step Mixing Time',
 'X_SV_Electrode_Mixing_Anode_Parameter Value_2 Step Input Material 6',
 'X_SV_Electrode_Mixing_Cathode_Parameter Value_2 Step Input Material 6',
 'X_SV_Electrode_Mixing_Anode_Parameter Value_1 Step Input Material 1 Set Weight']

In [15]:
cols_major_sv = [
    
]
# TODO : 주요 SV 받아서 변경 필요

cols_minor_sv = [

]

In [16]:
cols_big_y = [
    'Y_NFF_A',
    'Y_NFF_E',
    'Y_NFF_J',
    'Y_NFF_F',
    'Y_NFF_R',
    'Y_NFF_V'
]

In [17]:
cols_iqc = [x for x in data.columns if 'IQC' in x]
cols_iqc[:5]

['X_IQC_Assemble_Assembly_두께_1',
 'X_IQC_Assemble_Assembly_돌기부_외경',
 'X_IQC_Assemble_Assembly_중심부Hole_동심도',
 'X_IQC_Assemble_Assembly_외경',
 'X_IQC_Assemble_Assembly_두께_4']

# Preprocess data

In [18]:
# 0. Big Y 없는 row 제거
data = data[data['Y_NFF_A'].notnull()]

In [19]:
# 1. 중복 컬럼 제거
data = data.loc[:, ~data.columns.duplicated()]

In [20]:
# 2. datetime 처리
col_target_date = '06_Assembly_Finished Date'
data[col_target_date] = pd.to_datetime(data[col_target_date])
data['week'] = data[col_target_date].dt.isocalendar().week
data['date'] = data[col_target_date].dt.date

In [21]:
# 3. Category로 되어있는 Small y 인코딩
def encode_small_y(x) :
    # NG -> 불량 -> 1
    # OK -> 정상 -> 0
    if x == 'NG' :
        y = 1
    elif x == 'OK' : 
        y = 0 
    else : 
        y = x
    return y

for col in tqdm(cols_small_y) : 
    data[col] = data[col].apply(lambda x: encode_small_y(x))
    

100%|██████████| 134/134 [00:02<00:00, 46.06it/s]


In [22]:
# 4. 모든 Row가 결측치인 Small y drop

drop_cols = data[cols_small_y].columns[data[cols_small_y].isna().all()]
data = data.drop(columns=drop_cols)

cols_small_y_nan_dropped = [x for x in cols_small_y if x in data.columns]

'''
# cols_small_y weekly score에 있는것만 발라내기 
cols_small_y_nan_dropped = [x for x in cols_small_y_nan_dropped if x in weekly_score.columns]

data[cols_small_y_nan_dropped].isna().sum()
'''

'\n# cols_small_y weekly score에 있는것만 발라내기 \ncols_small_y_nan_dropped = [x for x in cols_small_y_nan_dropped if x in weekly_score.columns]\n\ndata[cols_small_y_nan_dropped].isna().sum()\n'

In [23]:
# 5. Small y 결측치 Historical cumalative mean으로 fill
data[cols_small_y_nan_dropped].describe()


,y_MES_Electrode_Coating_Anode_치수_Mismatch (Back),y_MES_Electrode_Coating_Cathode_외관_접힘 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back,y_MES_Electrode_Coating_Anode_절연 (Top),y_MES_Electrode_Coating_Anode_외관_표면 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Top,y_MES_Electrode_Coating_Cathode_치수_Loading (Top),y_MES_Electrode_Coating_Cathode_외관_표면 (Top),y_MES_Electrode_Coating_Cathode_치수_폭(Top),y_MES_Electrode_Coating_Anode_절연 (Back),...,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BUAGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_JLROTD_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BASGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BLCGAP_O1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_SLTPOS_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_CORRFM_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCAGAP_M2,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCSGAP_O1
count,57442.0,57018.0,8262.000000,57442.0,57442.000000,8262.000000,57018.000000,57018.000000,57018.0,57442.0,...,56899.000000,56899.0,56899.0,56899.000000,56899.000000,56899.0,56899.0,56899.0,56899.000000,56899.000000
mean,0.0,0.0,95.899229,0.0,0.615577,95.988441,0.073084,2.407410,0.0,0.0,...,0.309443,0.0,0.0,0.309443,0.309443,0.0,0.0,0.0,0.309443,0.309443
std,0.0,0.0,1.236320,0.0,9.103676,2.110083,1.709696,17.795562,0.0,0.0,...,0.462268,0.0,0.0,0.462268,0.462268,0.0,0.0,0.0,0.462268,0.462268
min,0.0,0.0,93.183333,0.0,0.000000,89.950000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000
25%,0.0,0.0,95.087500,0.0,0.000000,94.133333,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000
50%,0.0,0.0,95.783333,0.0,0.000000,96.166667,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000
75%,0.0,0.0,96.733333,0.0,0.000000,97.600000,0.000000,0.000000,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000
max,0.0,0.0,100.300000,0.0,300.000000,100.433333,40.100000,214.300000,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000


In [24]:
hist_mean = data[cols_small_y_nan_dropped].expanding().mean().shift(1)
data[cols_small_y_nan_dropped] = data[cols_small_y_nan_dropped].fillna(hist_mean)

In [25]:
data[cols_small_y_nan_dropped]

,y_MES_Electrode_Coating_Anode_치수_Mismatch (Back),y_MES_Electrode_Coating_Cathode_외관_접힘 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back,y_MES_Electrode_Coating_Anode_절연 (Top),y_MES_Electrode_Coating_Anode_외관_표면 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Top,y_MES_Electrode_Coating_Cathode_치수_Loading (Top),y_MES_Electrode_Coating_Cathode_외관_표면 (Top),y_MES_Electrode_Coating_Cathode_치수_폭(Top),y_MES_Electrode_Coating_Anode_절연 (Back),...,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BUAGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_JLROTD_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BASGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BLCGAP_O1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_SLTPOS_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_CORRFM_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCAGAP_M2,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCSGAP_O1
0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
2,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
3,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14567,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14568,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14569,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14570,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0


In [26]:
# 6. Historical mean으로 안채워지는거 0으로 fill
data[cols_small_y_nan_dropped] = data[cols_small_y_nan_dropped].fillna(0)

In [27]:
data[cols_small_y_nan_dropped]

,y_MES_Electrode_Coating_Anode_치수_Mismatch (Back),y_MES_Electrode_Coating_Cathode_외관_접힘 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back,y_MES_Electrode_Coating_Anode_절연 (Top),y_MES_Electrode_Coating_Anode_외관_표면 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Top,y_MES_Electrode_Coating_Cathode_치수_Loading (Top),y_MES_Electrode_Coating_Cathode_외관_표면 (Top),y_MES_Electrode_Coating_Cathode_치수_폭(Top),y_MES_Electrode_Coating_Anode_절연 (Back),...,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BUAGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_JLROTD_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BASGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BLCGAP_O1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_SLTPOS_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_CORRFM_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCAGAP_M2,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCSGAP_O1
0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
2,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0
3,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14567,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14568,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14569,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14570,0.0,0.0,95.899330,0.0,0.0,95.988843,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0


In [28]:
# 7. IQC 결측치 Historical cumalative mean으로 fill
cols_target = cols_iqc

data[cols_target].describe()

hist_mean = data[cols_target].expanding().mean().shift(1)
data[cols_target] = data[cols_target].fillna(hist_mean)

# Historical mean으로 안채워지는거 0으로 fill
data[cols_target] = data[cols_target].fillna(0)

data[cols_target]

,X_IQC_Assemble_Assembly_두께_1,X_IQC_Assemble_Assembly_돌기부_외경,X_IQC_Assemble_Assembly_중심부Hole_동심도,X_IQC_Assemble_Assembly_외경,X_IQC_Assemble_Assembly_두께_4,X_IQC_Assemble_Assembly_두께_2,X_IQC_Assemble_Assembly_중심부Hole_외경,X_IQC_Assemble_Assembly_두께_3,X_IQC_Assemble_Assembly_외관,X_IQC_Assemble_Assembly_Side_Hole_직경_2,...,X_IQC_Electrode_Coating_Anode_인장강도,X_IQC_Electrode_Coating_Cathode_인장강도,X_IQC_Electrode_Coating_Anode_Dyne_Test,X_IQC_Electrode_Coating_Cathode_Dyne_Test,X_IQC_Electrode_Coating_Anode_연신율,X_IQC_Electrode_Coating_Cathode_연신율,X_IQC_Electrode_Coating_Anode_표면조도_Drum_Ra,X_IQC_Electrode_Coating_Cathode_표면조도_Drum_Ra,X_IQC_Electrode_Coating_Anode_표면조도_Air_Ra,X_IQC_Electrode_Coating_Cathode_표면조도_Air_Ra
0,0.618141,45.318000,0.028750,41.428580,0.690934,0.246727,13.499750,0.281614,0.0,2.977556,...,0.00,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.000,0.0
1,0.593969,45.318000,0.028750,41.416891,0.705874,0.243909,13.499750,0.276783,0.0,2.977556,...,0.00,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.000,0.0
2,0.626175,45.320250,0.024250,41.735460,0.780391,0.247585,13.502750,0.275914,0.0,2.973000,...,0.00,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.000,0.0
3,0.601268,45.318800,0.026800,41.600597,0.760074,0.245199,13.500000,0.277656,0.0,2.972218,...,0.00,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.000,0.0
4,0.460907,45.279333,0.024667,40.386835,0.504042,0.227508,13.514667,0.247813,0.0,2.967895,...,0.00,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14567,0.610358,45.296429,0.028214,41.757182,0.820768,0.244664,13.500153,0.277876,0.0,2.968359,...,33.38,0.0,0.0,0.0,16.87,0.0,0.24,0.0,0.235,0.0
14568,0.633112,45.320250,0.024250,41.734844,0.771714,0.248399,13.502750,0.277188,0.0,2.973000,...,33.38,0.0,0.0,0.0,16.87,0.0,0.24,0.0,0.235,0.0
14569,0.736999,45.278095,0.024286,42.355159,0.891392,0.256109,13.512381,0.289099,0.0,2.967000,...,33.38,0.0,0.0,0.0,16.87,0.0,0.24,0.0,0.235,0.0
14570,0.479582,45.297333,0.032333,40.807354,0.597582,0.231191,13.498333,0.258297,0.0,2.980000,...,33.38,0.0,0.0,0.0,16.87,0.0,0.24,0.0,0.235,0.0


In [29]:
# 8. SV 결측치 Historical cumalative mean으로 fill
cols_target = cols_sv

data[cols_target].describe()

hist_mean = data[cols_target].expanding().mean().shift(1)
data[cols_target] = data[cols_target].fillna(hist_mean)

# Historical mean으로 안채워지는거 0으로 fill
data[cols_target] = data[cols_target].fillna(0)

data[cols_target]

,X_SV_Electrode_Mixing_Anode_Parameter Value_2-1 Step Mixing Time,X_SV_Electrode_Mixing_Cathode_Parameter Value_2-1 Step Mixing Time,X_SV_Electrode_Mixing_Anode_Parameter Value_2 Step Input Material 6,X_SV_Electrode_Mixing_Cathode_Parameter Value_2 Step Input Material 6,X_SV_Electrode_Mixing_Anode_Parameter Value_1 Step Input Material 1 Set Weight,X_SV_Electrode_Mixing_Cathode_Parameter Value_1 Step Input Material 1 Set Weight,X_SV_Electrode_Mixing_Anode_Parameter Value_5-1 Step H/D Mixing RPM,X_SV_Electrode_Mixing_Cathode_Parameter Value_5-1 Step H/D Mixing RPM,X_SV_Electrode_Mixing_Anode_Parameter Value_6 Step Input Material 4 Set Weight,X_SV_Electrode_Mixing_Cathode_Parameter Value_6 Step Input Material 4 Set Weight,...,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 9 위치,X_SV_Assemble_Assembly_Paramter Target Value_메인디스크 스텝 모터 속도 #9,X_SV_Assemble_Assembly_Paramter Target Value_메인디스크 스텝 모터 속도 #10,X_SV_Assemble_Assembly_Paramter Target Value_로드셀 #2 상한값,X_SV_Assemble_Assembly_Paramter Target Value_로드셀 #1 상한값,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 8 위치,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 2 위치,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 6 위치,X_SV_Assemble_Assembly_Paramter Target Value_메인디스크 스텝 모터 속도 #8,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 4 위치
0,40.0,0.0,0.0,0.0,475.5,0.000000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
1,40.0,0.0,0.0,0.0,475.5,0.000000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
2,40.0,0.0,0.0,0.0,472.1,0.000000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
3,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
4,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14567,40.0,50.0,0.0,0.0,473.8,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
14568,40.0,50.0,0.0,0.0,475.5,496.000303,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
14569,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
14570,40.0,50.0,0.0,0.0,475.5,496.000339,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17


# Show distribution

## Weekly score

In [30]:
if model_name == 'N32S' :
    weekly_score = (
        weekly_score
        .drop('date', axis=1)
        .set_index('week')
        .loc[:, cols_big_y+[x for x in cols_small_y_nan_dropped if x in weekly_score.columns]]
    )
elif model_name == 'N33Q' : 
    weekly_score = (
        data
        .groupby('week')
        [cols_big_y + cols_small_y_nan_dropped].mean()
    )
weekly_score

,Y_NFF_A,Y_NFF_E,Y_NFF_J,Y_NFF_F,Y_NFF_R,Y_NFF_V
week,,,,,,
5,0.997249,0.000435,0.000000,0.000007,0.000015,0.000000
6,0.996782,0.000564,0.000693,0.000000,0.000058,0.000005
7,0.997993,0.000480,0.000000,0.000000,0.000038,0.000008
8,0.998562,0.000534,0.000000,0.000000,0.000012,0.000008
9,0.996932,0.000457,0.000000,0.000000,0.000100,0.000002
10,0.998756,0.000398,0.000021,0.000000,0.000015,0.000045
11,0.998763,0.000364,0.000000,0.000000,0.000024,0.000011
12,0.999007,0.000293,0.000010,0.000032,0.000014,0.000008
13,0.998547,0.000369,0.000177,0.000022,0.000045,0.000010


In [31]:
# historical_small_y_target = (
#     weekly_score
#     #[[x for x in cols_small_y_nan_dropped if x in weekly_score.columns]]
#     .expanding().mean()
# )
# historical_small_y_target

In [32]:
historical_small_y_target = (
    data
    .loc[:, cols_small_y_nan_dropped + ['week']]
    .groupby('week')
    [cols_small_y_nan_dropped].mean()
)
historical_small_y_target

,y_MES_Electrode_Coating_Anode_치수_Mismatch (Back),y_MES_Electrode_Coating_Cathode_외관_접힘 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back,y_MES_Electrode_Coating_Anode_절연 (Top),y_MES_Electrode_Coating_Anode_외관_표면 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Top,y_MES_Electrode_Coating_Cathode_치수_Loading (Top),y_MES_Electrode_Coating_Cathode_외관_표면 (Top),y_MES_Electrode_Coating_Cathode_치수_폭(Top),y_MES_Electrode_Coating_Anode_절연 (Back),...,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BUAGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_JLROTD_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BASGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BLCGAP_O1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_SLTPOS_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_CORRFM_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCAGAP_M2,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCSGAP_O1
week,,,,,,,,,,,,,,,,,,,,,
5,0.0,0.0,95.948548,0.0,0.403769,96.176449,0.012339,2.737877,0.0,0.0,...,0.379310,0.0,0.0,0.379310,0.379310,0.0,0.0,0.0,0.379310,0.379310
6,0.0,0.0,95.947019,0.0,0.061450,96.220631,0.288145,0.088800,0.0,0.0,...,0.948872,0.0,0.0,0.948872,0.948872,0.0,0.0,0.0,0.948872,0.948872
7,0.0,0.0,95.948613,0.0,0.222917,96.268508,0.193984,2.828751,0.0,0.0,...,0.937294,0.0,0.0,0.937294,0.937294,0.0,0.0,0.0,0.937294,0.937294
8,0.0,0.0,95.936891,0.0,0.200223,96.374957,0.002802,4.624428,0.0,0.0,...,0.148787,0.0,0.0,0.148787,0.148787,0.0,0.0,0.0,0.148787,0.148787
9,0.0,0.0,95.887704,0.0,2.077787,96.129166,0.005980,5.448655,0.0,0.0,...,0.003815,0.0,0.0,0.003815,0.003815,0.0,0.0,0.0,0.003815,0.003815
10,0.0,0.0,95.775714,0.0,3.076384,95.753867,0.008479,4.471259,0.0,0.0,...,0.027993,0.0,0.0,0.027993,0.027993,0.0,0.0,0.0,0.027993,0.027993
11,0.0,0.0,95.826098,0.0,0.162464,95.735084,0.002976,1.745715,0.0,0.0,...,0.005050,0.0,0.0,0.005050,0.005050,0.0,0.0,0.0,0.005050,0.005050
12,0.0,0.0,95.848379,0.0,0.580042,95.874224,0.000834,0.242912,0.0,0.0,...,0.001733,0.0,0.0,0.001733,0.001733,0.0,0.0,0.0,0.001733,0.001733
13,0.0,0.0,95.820431,0.0,0.003172,95.691429,0.000766,2.295385,0.0,0.0,...,0.001701,0.0,0.0,0.001701,0.001701,0.0,0.0,0.0,0.001701,0.001701


In [33]:
historical_small_y_max = (
    data
    .loc[:, cols_small_y_nan_dropped + ['week']]
    .groupby('week')
    [cols_small_y_nan_dropped].max()
)
historical_small_y_max

,y_MES_Electrode_Coating_Anode_치수_Mismatch (Back),y_MES_Electrode_Coating_Cathode_외관_접힘 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back,y_MES_Electrode_Coating_Anode_절연 (Top),y_MES_Electrode_Coating_Anode_외관_표면 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Top,y_MES_Electrode_Coating_Cathode_치수_Loading (Top),y_MES_Electrode_Coating_Cathode_외관_표면 (Top),y_MES_Electrode_Coating_Cathode_치수_폭(Top),y_MES_Electrode_Coating_Anode_절연 (Back),...,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BUAGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_JLROTD_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BASGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BLCGAP_O1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_SLTPOS_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_CORRFM_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCAGAP_M2,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCSGAP_O1
week,,,,,,,,,,,,,,,,,,,,,
5,0.0,0.0,100.283333,0.0,30.000000,100.433333,0.232174,49.6,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000
6,0.0,0.0,100.300000,0.0,30.000000,99.933333,40.100000,46.5,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000
7,0.0,0.0,100.283333,0.0,50.000000,100.433333,40.100000,214.3,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000
8,0.0,0.0,98.250000,0.0,30.000000,99.900000,0.268792,202.9,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000
9,0.0,0.0,98.283333,0.0,30.000000,100.416667,0.278125,202.9,0.0,0.0,...,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,0.0,1.000000,1.000000
10,0.0,0.0,98.016667,0.0,300.000000,99.416667,0.314961,202.8,0.0,0.0,...,0.356164,0.0,0.0,0.356164,0.356164,0.0,0.0,0.0,0.356164,0.356164
11,0.0,0.0,97.316667,0.0,25.000000,96.906667,0.452542,121.0,0.0,0.0,...,0.331429,0.0,0.0,0.331429,0.331429,0.0,0.0,0.0,0.331429,0.331429
12,0.0,0.0,97.950000,0.0,50.000000,96.906667,0.482530,60.5,0.0,0.0,...,0.327061,0.0,0.0,0.327061,0.327061,0.0,0.0,0.0,0.327061,0.327061
13,0.0,0.0,97.950000,0.0,0.703535,97.057143,0.276207,196.1,0.0,0.0,...,0.326630,0.0,0.0,0.326630,0.326630,0.0,0.0,0.0,0.326630,0.326630


In [34]:
historical_small_y_min = (
    data
    .loc[:, cols_small_y_nan_dropped + ['week']]
    .groupby('week')
    [cols_small_y_nan_dropped].min()
)
historical_small_y_min

,y_MES_Electrode_Coating_Anode_치수_Mismatch (Back),y_MES_Electrode_Coating_Cathode_외관_접힘 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Back,y_MES_Electrode_Coating_Anode_절연 (Top),y_MES_Electrode_Coating_Anode_외관_표면 (Back),y_LQC_Electrode_Coating_Cathode_Fat_edge_Tab Top,y_MES_Electrode_Coating_Cathode_치수_Loading (Top),y_MES_Electrode_Coating_Cathode_외관_표면 (Top),y_MES_Electrode_Coating_Cathode_치수_폭(Top),y_MES_Electrode_Coating_Anode_절연 (Back),...,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BUAGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_JLROTD_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BASGAP_M1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BLCGAP_O1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_TABANG_A1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_SLTPOS_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_CORRFM_C1,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCAGAP_M2,y_SPCP_Assemble_Dimension_Winding_DIMENSION_BCSGAP_O1
week,,,,,,,,,,,,,,,,,,,,,
5,0.0,0.0,94.483333,0.0,0.0,95.910827,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,93.433333,0.0,0.0,93.233333,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,0.0,0.0,93.566667,0.0,0.0,92.733333,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## SV distribution

In [35]:
data_sv_count_by_week = (
    data
    .groupby('week')
    [cols_sv].count()
    #.fillna(0)
)
data_sv_count_by_week.max(axis=1)

week
5     1486
6     8381
7     9138
8     6262
9     4170
10    5317
11    6577
12    8209
13    8370
14    1053
dtype: int64

In [36]:
data_sv_last_by_week = (
    data
    .groupby('week')
    [cols_sv].last()
    .fillna(0)
)
data_sv_last_by_week

,X_SV_Electrode_Mixing_Anode_Parameter Value_2-1 Step Mixing Time,X_SV_Electrode_Mixing_Cathode_Parameter Value_2-1 Step Mixing Time,X_SV_Electrode_Mixing_Anode_Parameter Value_2 Step Input Material 6,X_SV_Electrode_Mixing_Cathode_Parameter Value_2 Step Input Material 6,X_SV_Electrode_Mixing_Anode_Parameter Value_1 Step Input Material 1 Set Weight,X_SV_Electrode_Mixing_Cathode_Parameter Value_1 Step Input Material 1 Set Weight,X_SV_Electrode_Mixing_Anode_Parameter Value_5-1 Step H/D Mixing RPM,X_SV_Electrode_Mixing_Cathode_Parameter Value_5-1 Step H/D Mixing RPM,X_SV_Electrode_Mixing_Anode_Parameter Value_6 Step Input Material 4 Set Weight,X_SV_Electrode_Mixing_Cathode_Parameter Value_6 Step Input Material 4 Set Weight,...,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 9 위치,X_SV_Assemble_Assembly_Paramter Target Value_메인디스크 스텝 모터 속도 #9,X_SV_Assemble_Assembly_Paramter Target Value_메인디스크 스텝 모터 속도 #10,X_SV_Assemble_Assembly_Paramter Target Value_로드셀 #2 상한값,X_SV_Assemble_Assembly_Paramter Target Value_로드셀 #1 상한값,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 8 위치,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 2 위치,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 6 위치,X_SV_Assemble_Assembly_Paramter Target Value_메인디스크 스텝 모터 속도 #8,X_SV_Assemble_Assembly_Paramter Target Value_CSZ 스핀들 4 위치
week,,,,,,,,,,,,,,,,,,,,,
5,40.0,50.0,0.0,0.0,472.1,480.900000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
6,40.0,50.0,0.0,0.0,475.5,496.000339,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
7,40.0,50.0,0.0,0.0,475.5,497.440000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
8,40.0,50.0,0.0,0.0,475.5,496.000303,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
9,40.0,50.0,0.0,0.0,472.1,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
10,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
11,40.0,50.0,0.0,0.0,473.8,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
12,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17
13,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,-0.13,150.0,150.0,0.0,2.5,-0.23,-0.25,-0.2,150.0,-0.17


In [37]:
model_path = FILEPATH_PCA_MODEL_SV
data_pca = data_sv_last_by_week

In [38]:
if SHOW_DISTRIBUTION :
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data_pca)


    if not os.path.exists(model_path) : 
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        with open(model_path, 'wb') as f : 
            dill.dump(pca, f)
    else : 
        with open(model_path, 'rb') as f : 
            pca = dill.load(f)
        X_pca = pca.transform((X_scaled))


In [39]:
if SHOW_DISTRIBUTION :
    print('SHOW DISTRIBUTION')
    data_pca_2dim = pd.DataFrame(
        X_pca,
        columns=['PC1', 'PC2'],
        index=getattr(data_pca, "index", None)
    )
    data_pca_2dim[cols_big_y + cols_small_y_nan_dropped] = weekly_score[cols_big_y + cols_small_y_nan_dropped]

    idx_col_y = 0
    col_y = (cols_big_y + cols_small_y_nan_dropped)[idx_col_y]

    x = data_pca_2dim['PC1'].values
    y = data_pca_2dim['PC2'].values
    score = (1-data_pca_2dim[col_y].values)
    week = data_pca_2dim.index.tolist()

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = Normalize(score.min(), score.max())

    lc = LineCollection(
        segments,
        cmap='viridis',
        norm=norm,
        linewidth=2
    )
    lc.set_array(score[:-1])

    fig, ax = plt.subplots(figsize=(7,6))
    ax.add_collection(lc)

    # Scatter
    sc = ax.scatter(
        x, y,
        c=score,
        cmap='viridis',
        norm=norm,
        s=30,
        edgecolor='k',
        zorder=3
    )

    # #1. 화살표 추가
    for i in range(len(x) - 1) : 
        arrow = FancyArrowPatch(
            (x[i], y[i]),
            (x[i+1], y[i+1]),
            arrowstyle="-|>",
            mutation_scale=10,
            color=plt.cm.viridis(norm(score[i])),
            linewidth=1.5,
            alpha=0.8,
            zorder=2,
        )
        ax.add_patch(arrow)

    # 2. 레이블 추가
    for real_idx, idx in enumerate(data_pca_2dim.index.tolist()) : 
        ax.text(
            x[real_idx],
            y[real_idx],
            f"Week : {week[real_idx]}, Score : {score[real_idx]}",
            fontsize=8,
            ha='left',
            va='bottom',
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc='white',
                alpha=0.7,
                ec='gray'
            )
        )

    ax.autoscale()

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'Defect rate {col_y}')

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Trajectory by week, by {col_y}")

    plt.tight_layout()
    plt.show()

## IQC distribution

In [40]:
data_iqc_mean_by_week = (
    data
    .groupby('week')
    [cols_iqc].mean()
    .fillna(0)
)
data_iqc_mean_by_week

,X_IQC_Assemble_Assembly_두께_1,X_IQC_Assemble_Assembly_돌기부_외경,X_IQC_Assemble_Assembly_중심부Hole_동심도,X_IQC_Assemble_Assembly_외경,X_IQC_Assemble_Assembly_두께_4,X_IQC_Assemble_Assembly_두께_2,X_IQC_Assemble_Assembly_중심부Hole_외경,X_IQC_Assemble_Assembly_두께_3,X_IQC_Assemble_Assembly_외관,X_IQC_Assemble_Assembly_Side_Hole_직경_2,...,X_IQC_Electrode_Coating_Anode_인장강도,X_IQC_Electrode_Coating_Cathode_인장강도,X_IQC_Electrode_Coating_Anode_Dyne_Test,X_IQC_Electrode_Coating_Cathode_Dyne_Test,X_IQC_Electrode_Coating_Anode_연신율,X_IQC_Electrode_Coating_Cathode_연신율,X_IQC_Electrode_Coating_Anode_표면조도_Drum_Ra,X_IQC_Electrode_Coating_Cathode_표면조도_Drum_Ra,X_IQC_Electrode_Coating_Anode_표면조도_Air_Ra,X_IQC_Electrode_Coating_Cathode_표면조도_Air_Ra
week,,,,,,,,,,,,,,,,,,,,,
5,0.603118,45.305966,0.027226,41.468895,0.711503,0.244478,13.502788,0.275370,0.0,2.973152,...,33.267685,0.0,0.0,0.0,16.813237,0.0,0.239192,0.0,0.234209,0.0
6,0.458838,45.304378,0.031918,40.716561,0.579058,0.228907,13.498476,0.253681,0.0,2.979645,...,33.304326,0.0,0.0,0.0,16.831755,0.0,0.239456,0.0,0.234467,0.0
7,0.652456,45.317367,0.028748,41.761642,0.772815,0.250493,13.499784,0.288184,0.0,2.977350,...,33.339818,0.0,0.0,0.0,16.849692,0.0,0.239711,0.0,0.234717,0.0
8,0.706337,45.320175,0.024400,42.051610,0.803828,0.256748,13.502650,0.290690,0.0,2.972930,...,33.305372,0.0,0.0,0.0,16.832284,0.0,0.239463,0.0,0.234475,0.0
9,0.821686,45.318936,0.024666,42.493600,0.827446,0.270052,13.500937,0.313443,0.0,2.971771,...,33.315962,0.0,0.0,0.0,16.837635,0.0,0.239540,0.0,0.234549,0.0
10,0.605145,45.318151,0.026016,41.475209,0.709572,0.245559,13.499460,0.276872,0.0,2.972121,...,33.348610,0.0,0.0,0.0,16.854136,0.0,0.239774,0.0,0.234779,0.0
11,0.610733,45.310406,0.028149,41.665341,0.776429,0.245587,13.499251,0.279633,0.0,2.972742,...,33.298796,0.0,0.0,0.0,16.828960,0.0,0.239416,0.0,0.234428,0.0
12,0.646215,45.286606,0.026075,41.835321,0.804555,0.247758,13.505933,0.279220,0.0,2.968139,...,33.327139,0.0,0.0,0.0,16.843284,0.0,0.239620,0.0,0.234628,0.0
13,0.507095,45.279711,0.024694,40.709774,0.562547,0.232208,13.513583,0.253528,0.0,2.967280,...,33.316191,0.0,0.0,0.0,16.837751,0.0,0.239541,0.0,0.234551,0.0


In [41]:
model_path = FILEPATH_PCA_MODEL_IQC
data_pca = data_iqc_mean_by_week

In [42]:
if SHOW_DISTRIBUTION :
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data_pca)


    if not os.path.exists(model_path) : 
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        with open(model_path, 'wb') as f : 
            dill.dump(pca, f)
    else : 
        with open(model_path, 'rb') as f : 
            pca = dill.load(f)
        X_pca = pca.transform((X_scaled))


In [43]:
if SHOW_DISTRIBUTION :
    print('SHOW DISTRIBUTION')
    data_pca_2dim = pd.DataFrame(
        X_pca,
        columns=['PC1', 'PC2'],
        index=getattr(data_pca, "index", None)
    )
    data_pca_2dim[cols_big_y + cols_small_y_nan_dropped] = weekly_score[cols_big_y + cols_small_y_nan_dropped]

    idx_col_y = 0
    col_y = (cols_big_y + cols_small_y_nan_dropped)[idx_col_y]

    x = data_pca_2dim['PC1'].values
    y = data_pca_2dim['PC2'].values
    score = (1-data_pca_2dim[col_y].values)
    week = data_pca_2dim.index.tolist()

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = Normalize(score.min(), score.max())

    lc = LineCollection(
        segments,
        cmap='viridis',
        norm=norm,
        linewidth=2
    )
    lc.set_array(score[:-1])

    fig, ax = plt.subplots(figsize=(7,6))
    ax.add_collection(lc)

    # Scatter
    sc = ax.scatter(
        x, y,
        c=score,
        cmap='viridis',
        norm=norm,
        s=30,
        edgecolor='k',
        zorder=3
    )

    # #1. 화살표 추가
    for i in range(len(x) - 1) : 
        arrow = FancyArrowPatch(
            (x[i], y[i]),
            (x[i+1], y[i+1]),
            arrowstyle="-|>",
            mutation_scale=10,
            color=plt.cm.viridis(norm(score[i])),
            linewidth=1.5,
            alpha=0.8,
            zorder=2,
        )
        ax.add_patch(arrow)

    # 2. 레이블 추가
    for real_idx, idx in enumerate(data_pca_2dim.index.tolist()) : 
        ax.text(
            x[real_idx],
            y[real_idx],
            f"Week : {week[real_idx]}, Score : {score[real_idx]}",
            fontsize=8,
            ha='left',
            va='bottom',
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc='white',
                alpha=0.7,
                ec='gray'
            )
        )

    ax.autoscale()

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'Defect rate {col_y}')

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Trajectory by week, by {col_y}")

    plt.tight_layout()
    plt.show()

## SV + IQC distribution

In [44]:
data_sv_last_iqc_mean_by_week = pd.concat([
    data_sv_last_by_week,
    data_iqc_mean_by_week
], axis=1)
data_sv_last_iqc_mean_by_week

,X_SV_Electrode_Mixing_Anode_Parameter Value_2-1 Step Mixing Time,X_SV_Electrode_Mixing_Cathode_Parameter Value_2-1 Step Mixing Time,X_SV_Electrode_Mixing_Anode_Parameter Value_2 Step Input Material 6,X_SV_Electrode_Mixing_Cathode_Parameter Value_2 Step Input Material 6,X_SV_Electrode_Mixing_Anode_Parameter Value_1 Step Input Material 1 Set Weight,X_SV_Electrode_Mixing_Cathode_Parameter Value_1 Step Input Material 1 Set Weight,X_SV_Electrode_Mixing_Anode_Parameter Value_5-1 Step H/D Mixing RPM,X_SV_Electrode_Mixing_Cathode_Parameter Value_5-1 Step H/D Mixing RPM,X_SV_Electrode_Mixing_Anode_Parameter Value_6 Step Input Material 4 Set Weight,X_SV_Electrode_Mixing_Cathode_Parameter Value_6 Step Input Material 4 Set Weight,...,X_IQC_Electrode_Coating_Anode_인장강도,X_IQC_Electrode_Coating_Cathode_인장강도,X_IQC_Electrode_Coating_Anode_Dyne_Test,X_IQC_Electrode_Coating_Cathode_Dyne_Test,X_IQC_Electrode_Coating_Anode_연신율,X_IQC_Electrode_Coating_Cathode_연신율,X_IQC_Electrode_Coating_Anode_표면조도_Drum_Ra,X_IQC_Electrode_Coating_Cathode_표면조도_Drum_Ra,X_IQC_Electrode_Coating_Anode_표면조도_Air_Ra,X_IQC_Electrode_Coating_Cathode_표면조도_Air_Ra
week,,,,,,,,,,,,,,,,,,,,,
5,40.0,50.0,0.0,0.0,472.1,480.900000,0.0,0.0,0.0,0.0,...,33.267685,0.0,0.0,0.0,16.813237,0.0,0.239192,0.0,0.234209,0.0
6,40.0,50.0,0.0,0.0,475.5,496.000339,0.0,0.0,0.0,0.0,...,33.304326,0.0,0.0,0.0,16.831755,0.0,0.239456,0.0,0.234467,0.0
7,40.0,50.0,0.0,0.0,475.5,497.440000,0.0,0.0,0.0,0.0,...,33.339818,0.0,0.0,0.0,16.849692,0.0,0.239711,0.0,0.234717,0.0
8,40.0,50.0,0.0,0.0,475.5,496.000303,0.0,0.0,0.0,0.0,...,33.305372,0.0,0.0,0.0,16.832284,0.0,0.239463,0.0,0.234475,0.0
9,40.0,50.0,0.0,0.0,472.1,497.400000,0.0,0.0,0.0,0.0,...,33.315962,0.0,0.0,0.0,16.837635,0.0,0.239540,0.0,0.234549,0.0
10,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,33.348610,0.0,0.0,0.0,16.854136,0.0,0.239774,0.0,0.234779,0.0
11,40.0,50.0,0.0,0.0,473.8,497.400000,0.0,0.0,0.0,0.0,...,33.298796,0.0,0.0,0.0,16.828960,0.0,0.239416,0.0,0.234428,0.0
12,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,33.327139,0.0,0.0,0.0,16.843284,0.0,0.239620,0.0,0.234628,0.0
13,40.0,50.0,0.0,0.0,475.5,497.400000,0.0,0.0,0.0,0.0,...,33.316191,0.0,0.0,0.0,16.837751,0.0,0.239541,0.0,0.234551,0.0


In [45]:
model_path = FILEPATH_PCA_MODEL_SV_IQC
data_pca = data_sv_last_iqc_mean_by_week

In [46]:
if SHOW_DISTRIBUTION :
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data_pca)


    if not os.path.exists(model_path) : 
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        with open(model_path, 'wb') as f : 
            dill.dump(pca, f)
    else : 
        with open(model_path, 'rb') as f : 
            pca = dill.load(f)
        X_pca = pca.transform((X_scaled))


In [47]:
if SHOW_DISTRIBUTION :
    print('SHOW DISTRIBUTION')
    data_pca_2dim = pd.DataFrame(
        X_pca,
        columns=['PC1', 'PC2'],
        index=getattr(data_pca, "index", None)
    )
    data_pca_2dim[cols_big_y + cols_small_y_nan_dropped] = weekly_score[cols_big_y + cols_small_y_nan_dropped]

    idx_col_y = 0
    col_y = (cols_big_y + cols_small_y_nan_dropped)[idx_col_y]

    x = data_pca_2dim['PC1'].values
    y = data_pca_2dim['PC2'].values
    score = (1-data_pca_2dim[col_y].values)
    week = data_pca_2dim.index.tolist()

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = Normalize(score.min(), score.max())

    lc = LineCollection(
        segments,
        cmap='viridis',
        norm=norm,
        linewidth=2
    )
    lc.set_array(score[:-1])

    fig, ax = plt.subplots(figsize=(7,6))
    ax.add_collection(lc)

    # Scatter
    sc = ax.scatter(
        x, y,
        c=score,
        cmap='viridis',
        norm=norm,
        s=30,
        edgecolor='k',
        zorder=3
    )

    # #1. 화살표 추가
    for i in range(len(x) - 1) : 
        arrow = FancyArrowPatch(
            (x[i], y[i]),
            (x[i+1], y[i+1]),
            arrowstyle="-|>",
            mutation_scale=10,
            color=plt.cm.viridis(norm(score[i])),
            linewidth=1.5,
            alpha=0.8,
            zorder=2,
        )
        ax.add_patch(arrow)

    # 2. 레이블 추가
    for real_idx, idx in enumerate(data_pca_2dim.index.tolist()) : 
        ax.text(
            x[real_idx],
            y[real_idx],
            f"Week : {week[real_idx]}, Score : {score[real_idx]}",
            fontsize=8,
            ha='left',
            va='bottom',
            bbox=dict(
                boxstyle="round,pad=0.2",
                fc='white',
                alpha=0.7,
                ec='gray'
            )
        )

    ax.autoscale()

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f'Defect rate {col_y}')

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Trajectory by week, by {col_y}")

    plt.tight_layout()
    plt.show()

# Model cross validation

In [48]:
def clean_col_name(col):
    col=str(col)
    col=re.sub(R"_+", "_",col)
    col=re.sub(r"[^가-힣a-zA-Z0-9_]", "_", col)
    return col

In [49]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    score = []
    for week in tqdm(data_sv_last_by_week.index[:-1]) : 
        filename_result = f"results/result_cv_{model_version}_{model_name}_maximize_n_trials_{n_trials}_week_{week}.dill"
        filename_model = f"results/model_{model_version}_{model_name}_week_{week}.dill"
        if not os.path.exists(filename_result) : 
            gc.collect()
            # 1. 데이터 Split
            ## TODO : [ ] 2단계 모델 스킴으로 바꿔야 됨
            '''
            x_train = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_sv+cols_iqc]
            )
            x_test = (
                data
                .loc[lambda x : x['week']>week]
                .loc[lambda x : x['week']<=week+1]
                .loc[:, cols_sv+cols_iqc]
            )
            y_train = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_big_y+cols_small_y_nan_dropped]
            )
            y_test = (
                data
                .loc[lambda x : x['week']>week]
                .loc[lambda x : x['week']<=week+1]
                .loc[:, cols_big_y+cols_small_y_nan_dropped]
            )
            '''
            x = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_sv+cols_iqc]
            )
            y = (
                data
                .loc[lambda x : x['week']<=week]
                .loc[:, cols_big_y+cols_small_y_nan_dropped]
            )

            x_train, x_test, y_train, y_test = train_test_split(
                x,
                y,
                test_size=0.2,
                random_state=42,
                shuffle=True
            )


            x_train = x_train.astype('float32')
            y_train = y_train.astype('float32')
            x_test = x_test.astype('float32')
            y_test = y_test.astype('float32')

            print('reg = xgb.XGBRegressor')
            reg = CascadeModel(
                cols_small_y=cols_small_y_nan_dropped,
                cols_big_y=cols_big_y
            )

            reg.fit(x_train, y_train)

            y_pred_dict = reg.predict(x_test)
            y_pred_big_y = y_pred_dict['big_y']
            y_pred_small_y = y_pred_dict['small_y']



            y_pred_dict = reg.predict(x_test)
            y_pred_big_y = y_pred_dict['big_y']
            y_pred_small_y = y_pred_dict['small_y']

            # TODO : [ ] y_pred안에 cols_big_y랑 cols_small_y가 다 담겨져 나와야 함

            # TODO : [ ] metric 계산할때 따로 계산하든지 concat 처리가 필요
            y_pred = pd.concat([y_pred_big_y, y_pred_small_y], axis=1)

            metrics = []
            for idx, col in enumerate(y_test.columns) :
                y_test_by_column = y_test[col]
                y_pred_by_column = y_pred[clean_col_name(col)]

                try : 
                    roc_auc_by_column = roc_auc_score(y_test_by_column, y_pred_by_column)
                except : 
                    roc_auc_by_column = np.nan

                try : 
                    rmse_by_column =root_mean_sqaured_error(y_test_by_column, y_pred_by_column)
                except : 
                    rmse_by_column = np.nan
                metrics.append({
                    'col' : col,
                    'auroc' : roc_auc_by_column,
                    'rmse' : rmse_by_column
                })

            result = {
                'week' : week,
                'train_start' : data.loc[lambda x  :x['week']<=week][col_target_date].min(),
                'train_end' : data.loc[lambda x : x['week']<=week][col_target_date].max(),
                'test_start' : data.loc[lambda x : x['week']>week].loc[lambda x : x['week']<=week+1][col_target_date].min(),
                'test_end' : data.loc[lambda x : x['week']>week].loc[lambda x : x['week']<=week+1][col_target_date].max(),
                'metrics' : metrics,
                'artifacts' : {
                    'x_train' : x_train,
                    'x_test' : x_test,
                    'y_train' : y_train,
                    'y_test' : y_test,
                    'y_pred' : y_pred
                }
                        }
            with open(filename_result, 'wb') as f : 
                dill.dump(result, f)
            with open(filename_model, 'wb') as f : 
                dill.dump(reg, f)
            
        else : 
            with open(filename_result, 'rb') as f : 
                result = dill.load(f)    
    score.append(result)
       

PERFORM_CROSS_VALIDATION


  0%|          | 0/9 [00:00<?, ?it/s]

reg = xgb.XGBRegressor
(298, 4335)
(298, 4335)


 11%|█         | 1/9 [4:43:57<37:51:40, 17037.54s/it]

reg = xgb.XGBRegressor
(1974, 4335)
(1974, 4335)


 22%|██▏       | 2/9 [10:31:54<37:31:14, 19296.31s/it]

reg = xgb.XGBRegressor


# Visualize model CV result

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    result_dict = []

    for week in tqdm(data_sv_last_by_week.index[:-1]) : 
        filename_result = f"results/result_cv_{model_version}_{model_name}_maximize_n_trials_{n_trials}_week_{week}.dill"
        with open(filename_result, 'rb') as f : 
            result = dill.load(f)

        y_test = result['artifacts']['y_test']
        y_pred = pd.DataFrame(result['artifacts']['y_pred'])
        y_pred.columns = y_test.columns


        for col in y_test.columns : 
            try : 
                roc_auc = roc_auc_score(y_test[col], y_pred[col])
                rmse = np.nan
            except : 
                rmse = root_mean_squared_error(y_test[col], y_pred[col])
                roc_auc = np.nan
            result_dict.append({
                'week' : week,
                'col' : col,
                'roc_auc' : roc_auc,
                'rmse' : rmse
            })

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    score_df =(
        pd.DataFrame(result_dict)
        .pivot(
            index='week',
            columns='col',
            values=['roc_auc', 'rmse']
        )
    )
    score_df

In [ ]:
if PERFORM_CROSS_VALIDATION : 
    print('PERFORM_CROSS_VALIDATION')
    filename_score = f"results/result_{model_version}_{model_name}.csv"
    score_df.to_csv(filename_score)

# Run backtest

In [ ]:
data_sv_last_by_week

In [ ]:
if PERFORM_BACKTEST : 
    print('PERFORM_BACKTEST')

In [ ]:
historical_small_y_max

In [ ]:

for week in tqdm(data_sv_last_by_week.index) : 
    filename_backtest_result = f"results/result_backtest_{model_version}_{model_name}_n_trials_{n_trials}_week_{week}.dill"
    #filename_optuna_result = f"results/result_backtest_optuna_{model_version}_{model_name}_n_trials_{n_trials}_week_{week}.dill"
    filename_model = f"results/model_{model_version}_{model_name}_week_{week}.dill"
    if not os.path.exists(filename_backtest_result) : 
        gc.collect()
        # 1-1. 모델 불러오기
        print('# 1-1. 모델 불러오기')
        if os.path.exists(filename_model) :
            with open(filename_model, 'rb') as f : 
                model = dill.load(f)

        # 2. 초기값 설정
        print('# 2. 초기값 설정')
        x0 = (
            data
            .loc[lambda x : x['week']<=week]
            .loc[:, cols_sv+cols_iqc]
            .iloc[-1]
        )

        # 3. 상/하한 설정
        print('# 3. 상/하한 설정')
        target_data = data_sv_last_by_week.loc[lambda x : x.index <= week]
        bounds = {
            col : (
                (
                    target_data[col].min() 
                    - abs(target_data[col].min() )*lb_margin
                ), 
                (
                    target_data[col].max() 
                    + abs(target_data[col].max())*ub_margin
            )
            )
            for col in data_sv_last_by_week.columns
        }
        
        # 4. Constraint로 넣을 컬럼 설정
        ## 변동성 없는 컬럼 그대로 사용, IQC 그대로 사용 
        print('# 4. Constraint로 넣을 컬럼 설정')   
        cols_low_deviation = (
            (
            data_sv_last_by_week.nunique()
            [data_sv_last_by_week.nunique()<=threshold_constraint]
            .index.tolist()
            )
        )

        cols_iqc = [x for x in data.columns if 'IQC' in x]

        cols_to_fix = cols_low_deviation + cols_iqc

        # 5. 목적함수 정의
        print('# 5. 목적함수 정의')
        def objective(trial):
            gc.collect()
            print(f"Running trial {trial.number} in {threading.current_thread()}")
            # 5-1. 초기값 설정
            print('# 5-1. 초기값 설정')
            x_new = x0.copy()

            # 5-2. Constraint 제외하고 변경할 값 제안
            print('# 5-2. Constraint 제외하고 변경할 값 제안')
            opt_cols = list(set(cols_sv) - set(cols_to_fix))

            for col in opt_cols : 
                low, high = bounds[col]
                x_new[col] = trial.suggest_float(col, low, high)

            X = pd.DataFrame([x_new])

            y_pred_dict = model.predict(X)
            y_pred_big_y = y_pred_dict['big_y']
            y_pred_small_y = y_pred_dict['small_y']

            y_pred = pd.concat([y_pred_big_y, y_pred_small_y], axis=1)

            y_pred.columns = y_test.columns
            
            # big y
            # A는 1-A 해서 minimize
            y_pred['Y_NFF_A'] = 1- y_pred['Y_NFF_A']
            mean_big_y = abs(y_pred[cols_big_y].mean().mean())

            cols_small_y_nan_dropped_result = [x for x in cols_small_y_nan_dropped if x in y_pred.columns]
            mean_small_y = np.abs((
                (
                    y_pred[cols_small_y_nan_dropped_result].values 
                    - historical_small_y_target.loc[lambda x : x.index==week][cols_small_y_nan_dropped_result].values
                ) 
                /
                (
                    historical_small_y_target.loc[lambda x : x.index==week][cols_small_y_nan_dropped_result].values + 0.1
                )
            )
            .mean()
            )
            #print('mean_Small_y')
            #display(mean_small_y)
            # 나머지는 그냥 minimize

            is_out_of_range_max = (
                y_pred[cols_small_y_nan_dropped_result].values > historical_small_y_max.loc[lambda x : x.index==week][cols_small_y_nan_dropped_result].values
               
            )
            is_out_of_range_min = (
                y_pred[cols_small_y_nan_dropped_result].values < historical_small_y_min.loc[lambda x : x.index==week][cols_small_y_nan_dropped_result].values
            )

            is_out_of_range = is_out_of_range_max.any() | is_out_of_range_min.any()

            if is_out_of_range.any() : 
                raise optuna.TrialPruned(
                    f"{is_out_of_range.sum()} predictions are outside historical range"
                )


            target = mean_big_y + mean_small_y
            return target

        # 6. 최적화
        print('# 6. 최적화')
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, 
        n_trials=n_trials, 
        show_progress_bar=False, 
        n_jobs=-1
        )

        # with open(filename_optuna_result, 'wb') as f : 
        #     dill.dump(study, f)

        # 7. 결과 저장
        print('# 7. 결과 저장')
        backtest_result = {
            'best_params' : pd.DataFrame([study.best_params]),
            'study' : study
        }
        with open(filename_backtest_result, 'wb') as f : 
            dill.dump(backtest_result, f)

# Visualize backtest result

In [ ]:
week = data_sv_last_by_week.index[0]

filename_backtest_result = f"results/result_backtest_{model_version}_{model_name}_n_trials_{n_trials}_week_{week}.dill"
with open(filename_backtest_result, 'rb') as f : 
        backtest_result = dill.load(f)
study = backtest_result['study']
#print(study)
plot_optimization_history(study)

In [ ]:
week = data_sv_last_by_week.index[1]

filename_backtest_result = f"results/result_backtest_{model_version}_{model_name}_n_trials_{n_trials}_week_{week}.dill"
with open(filename_backtest_result, 'rb') as f : 
        backtest_result = dill.load(f)
study = backtest_result['study']
#print(study)
plot_optimization_history(study)

In [ ]:
for week in data_sv_last_by_week.index : 

    filename_backtest_result = f"results/result_backtest_{model_version}_{model_name}_n_trials_{n_trials}_week_{week}.dill"
    with open(filename_backtest_result, 'rb') as f : 
            backtest_result = dill.load(f)
    study = backtest_result['study']
    #print(study)
    fig = plot_optimization_history(study)
    show(fig)

In [ ]:
filelist = [x for x in 
os.listdir('results')
if f"result_backtest_{model_version}_{model_name}_n_trials_{n_trials}" in x]
filelist.sort()

In [ ]:
data_sv_result = []

for filename in filelist :
    tmp = pd.read_csv(f"results/{filename}").drop('Unnamed: 0', axis=1)
    data_sv_result.append(tmp['backtest_result'])

data_sv_result = pd.concat(data_sv_result)
data_sv_result.index = data_sv_last_by_week.index + 100

data_sv_result

In [ ]:
data_sv_mean_by_week_with_param = pd.concat([data_sv_last_by_week, data_sv_result], axis=0)
data_sv_mean_by_week_with_param = data_sv_mean_by_week_with_param
for col in data_sv_mean_by_week_with_param.columns : 
    data_sv_mean_by_week_with_param[col] = data_sv_mean_by_week_with_param[col].ffill()
data_sv_mean_by_week_with_param

In [ ]:
data_sv_mean_by_week = data_sv_mean_by_week_with_param

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_sv_mean_by_week)

filepath_pca_model = FILEPATH_PCA_MODEL_SV

if not os.path.exists(filepath_pca_model) : 
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    with open(filepath_pca_model, 'wb') as f : 
        dill.dump(pca, f)
else : 
    with open(filepath_pca_model, 'rb') as f : 
        pca = dill.load(f)
    X_pca = pca.transform((X_scaled))


In [ ]:
target_col_visualize = 'Y_NFF_A'

In [ ]:
data_sv_pca = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2'],
    index=getattr(data_sv_mean_by_week, "index", None)
)
data_sv_pca[target_col_visualize] = weekly_score[target_col_visualize]

In [ ]:
data_sv_pca

In [ ]:
x = data_sv_pca['PC1'].values
y = data_sv_pca['PC2'].values
score = (1-data_sv_pca[target_col_visualize].values)
week = data_sv_pca.index.tolist()

In [ ]:


points = np.array([x, y]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

norm = Normalize(score.min(), score.max())

lc = LineCollection(
    segments,
    cmap='viridis',
    norm=norm,
    linewidth=2
)
lc.set_array(score[:-1])

fig, ax = plt.subplots(figsize=(7,6))
ax.add_collection(lc)

# Scatter
sc = ax.scatter(
    x, y,
    c=score,
    cmap='viridis',
    norm=norm,
    s=30,
    edgecolor='k',
    zorder=3
)

# #1. 화살표 추가
for i in range(len(x) - 1) : 
    arrow = FancyArrowPatch(
        (x[i], y[i]),
        (x[i+1], y[i+1]),
        arrowstyle="-|>",
        mutation_scale=10,
        color=plt.cm.viridis(norm(score[i])),
        linewidth=1.5,
        alpha=0.8,
        zorder=2,
    )
    ax.add_patch(arrow)

# 2. 레이블 추가
for real_idx, idx in enumerate(data_sv_pca.index.tolist()) : 
    ax.text(
        x[real_idx],
        y[real_idx],
        f"Week : {week[real_idx]}, Score : {score[real_idx]}",
        fontsize=8,
        ha='left',
        va='bottom',
        bbox=dict(
            boxstyle="round,pad=0.2",
            fc='white',
            alpha=0.7,
            ec='gray'
        )
    )

ax.autoscale()

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Defect rate')

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Trajectory by date")

plt.tight_layout()
plt.show()